# Encoding Masterclass — every technique on real banking and Titanic data

01 Core Python · 02 Pandas · 03 Cleaning · 04 Transformation · **▶ 05 Feature Engineering** · 06 Regression · 07 Model Prep · 08 Case Studies

`05_Feature_Engineering/03_encoding_masterclass.ipynb`

---

### In one paragraph (no jargon)

The deep-dive companion to section 04's catalogue. Same techniques, but applied to datasets with real quirks — missing categories, heavy imbalance, and columns where the encoding choice visibly changes the result. Work through this one when you want to see *why* the decision rule in section 04 says what it says.

### After this notebook you can

- Compare every encoding technique on the same real column
- See the effect of each on column count, model input and interpretability
- Apply target encoding without leaking the target


### What's inside

1. Setup
2. The techniques, one by one
3. Side-by-side comparison
4. Leakage-safe target encoding
5. Exam quick-reference

---

> **▶ Runs on its own.** The next cell is the only setup you need. It imports the
> libraries and loads the data. If the `datasets/` folder isn't where it expects,
> it rebuilds an equivalent dataset in memory so **every cell below still runs** —
> handy if you copy this single `.ipynb` somewhere else.
>
> **▶ Reading the cells.** Code comments explain *what the line does*; the text
> blocks explain *why you'd do it*. Look for these markers:
> `# WHAT:` a plain-English translation · `# WHY:` the reason it matters ·
> `# 🔧 CHANGE THIS:` the knob to turn when the exam question differs ·
> **⚡ Beyond the syllabus** = optional, higher-mark techniques.

### Jargon buster

| Term | Plain English |
|---|---|
| **Feature** | One column used as an input to a model. |
| **Feature engineering** | Creating better columns from the ones you have — the step that usually improves a model more than changing the algorithm. |
| **Feature extraction** | Pulling a new column out of an existing one: month out of a date, title out of a name. |
| **Feature splitting** | Breaking one column into several: `"Yangon(Local)"` → city + type. |
| **Derived feature** | A column computed from others: profit = revenue − cost. |
| **Domain knowledge** | Knowing what the numbers mean in the business. This is what makes a good feature; no algorithm can supply it. |

In [1]:
# =============================================================================
# SETUP — run this cell first. It is the only cell with dependencies.
# =============================================================================
# WHAT: `import` pulls in code other people have written so we don't rewrite it.
#       The `as pd` part is a nickname, so we can type `pd` instead of `pandas`.
import pandas as pd          # tables of data (think: Excel, but programmable)
import numpy as np           # fast maths on whole columns at once
import matplotlib.pyplot as plt   # charts
import warnings

warnings.filterwarnings('ignore')          # hide version-upgrade notices, keeps output readable
pd.set_option('display.max_columns', 50)   # don't hide columns behind "..."
pd.set_option('display.width', 160)
import os
import seaborn as sns
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------------------------
# `category_encoders` provides Binary, Hashing and Target encoders. It is an
# optional library, so if it isn't installed we supply equivalents written in
# plain pandas below. Seeing the implementation is arguably more instructive
# than importing it — each is only a few lines.
# ---------------------------------------------------------------------------
try:
    import category_encoders as ce
    print("Using the category_encoders library.")
except ImportError:
    print("category_encoders not installed - using the built-in equivalents below.")

    class _BinaryEncoder:
        """Label-encode, then write the label number in binary across several columns.

        WHY: k categories need only log2(k) columns instead of one-hot's k.
        """
        def __init__(self, cols=None, **kw):
            self.cols, self.maps_ = cols, {}

        def fit(self, X, y=None):
            X = pd.DataFrame(X).copy()
            for col in (self.cols or X.columns):
                self.maps_[col] = {v: i for i, v in enumerate(sorted(X[col].astype(str).unique()))}
            return self

        def transform(self, X):
            X = pd.DataFrame(X).copy()
            out = pd.DataFrame(index=X.index)
            for col, mapping in self.maps_.items():
                codes = X[col].astype(str).map(mapping).fillna(0).astype(int).to_numpy()
                width = max(1, int(np.ceil(np.log2(max(len(mapping), 2)))))
                for bit in range(width):
                    out[f"{col}_{bit}"] = np.bitwise_and(np.right_shift(codes, bit), 1)
            return out

        def fit_transform(self, X, y=None):
            return self.fit(X, y).transform(X)

    class _HashingEncoder:
        """Hash each category into one of n fixed buckets.

        WHY: the output width is fixed in advance, however many categories appear -
        so an unseen category at prediction time can never break anything.
        COST: two categories can land in the same bucket (a 'collision').
        """
        def __init__(self, n_components=8, cols=None, **kw):
            self.n, self.cols = n_components, cols

        def fit(self, X, y=None):
            return self

        def transform(self, X):
            X = pd.DataFrame(X).copy()
            out = pd.DataFrame(0, index=X.index, columns=[f"col_{i}" for i in range(self.n)])
            for col in (self.cols or X.columns):
                for i, value in enumerate(X[col].astype(str)):
                    out.iloc[i, hash(value) % self.n] += 1
            return out

        def fit_transform(self, X, y=None):
            return self.fit(X, y).transform(X)

    class _TargetEncoder:
        """Replace each category with the mean of the target for that category,
        blended toward the overall mean so rare categories aren't trusted too far."""
        def __init__(self, cols=None, smoothing=10.0, **kw):
            self.cols, self.smoothing, self.maps_, self.prior_ = cols, smoothing, {}, 0.0

        def fit(self, X, y):
            X, y = pd.DataFrame(X).copy(), pd.Series(y).reset_index(drop=True)
            self.prior_ = y.mean()
            for col in (self.cols or X.columns):
                grouped = y.groupby(X[col].astype(str).reset_index(drop=True))
                counts, means = grouped.count(), grouped.mean()
                # smoothing: small groups are pulled toward the overall mean
                self.maps_[col] = ((counts * means + self.smoothing * self.prior_)
                                   / (counts + self.smoothing))
            return self

        def transform(self, X):
            X = pd.DataFrame(X).copy()
            out = pd.DataFrame(index=X.index)
            for col, mapping in self.maps_.items():
                out[col] = X[col].astype(str).map(mapping).fillna(self.prior_)
            return out

        def fit_transform(self, X, y=None):
            return self.fit(X, y).transform(X)

    class _CE:
        BinaryEncoder = _BinaryEncoder
        HashingEncoder = _HashingEncoder
        TargetEncoder = _TargetEncoder

    ce = _CE()

def find_datasets_folder(start=None):
    """Walk upwards from this notebook looking for the shared `datasets/` folder.

    WHY: it means the notebook works whether you opened it from its own folder,
    from the top of the notes, or from anywhere else on your machine.
    """
    here = os.path.abspath(start or os.getcwd())
    for _ in range(6):                       # look up to 6 folders up
        candidate = os.path.join(here, 'datasets')
        if os.path.isdir(candidate):
            return candidate
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return None

def dataset_path(filename, rebuild=None):
    """Return a real path to `filename`, materialising a temp copy if it's missing.

    WHY: a few pandas tools (pd.ExcelFile, pd.read_sql) need an actual file path
         rather than a DataFrame, so the fallback has to be written to disk.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if rebuild is None:
        raise FileNotFoundError(filename)
    import tempfile
    tmp = os.path.join(tempfile.mkdtemp(prefix='bda_'), filename)
    frame = rebuild()
    (frame.to_excel(tmp, index=False) if filename.lower().endswith(('.xlsx', '.xls'))
     else frame.to_csv(tmp, index=False))
    print(f"'{filename}' not found -> wrote a rebuilt copy to {tmp}")
    return tmp

def load_data(filename, rebuild=None, **read_kwargs):
    """Load `filename` from the shared datasets folder, or rebuild it in memory.

    WHAT: tries to read the real file; if it can't find it, calls `rebuild()`
          which recreates a dataset with the same columns and behaviour.
    WHY:  guarantees this notebook runs even if the CSV goes missing.
    """
    folder = find_datasets_folder()
    if folder:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            reader = pd.read_excel if filename.lower().endswith(('.xlsx', '.xls')) else pd.read_csv
            print(f"Loaded '{filename}' from {folder}")
            return reader(path, **read_kwargs)
    if rebuild is None:
        raise FileNotFoundError(f"Could not find {filename} and no fallback was supplied.")
    print(f"'{filename}' not found on disk -> rebuilding an equivalent dataset in memory.")
    built = rebuild()
    if 'chunksize' in read_kwargs:            # keep chunked reads working on the fallback path
        size = read_kwargs['chunksize']
        return (built.iloc[i:i + size] for i in range(0, len(built), size))
    return built

def rebuild_banking():
    """Rebuild a dataset statistically equivalent to banking.csv (41,199 rows).

    Same columns, same types, same ranges and category mix, so every cell
    below still runs if the original file is missing."""
    rng = np.random.default_rng(42)
    n = 41199
    frame = pd.DataFrame({
        'age': rng.normal(40.02, 10.43, n).clip(1, 104).round().astype(int),
        'job': rng.choice(['admin.', 'blue-collar', 'technician', 'services', 'management', 'retired', 'entrepreneur', 'self-employed', 'housemaid', 'unemployed', 'student', 'unknown'], n, p=[0.253, 0.2246, 0.1637, 0.0964, 0.071, 0.0418, 0.0353, 0.0345, 0.0258, 0.0246, 0.0213, 0.008]),
        'marital': rng.choice(['married', 'single', 'divorced', 'unknown'], n, p=[0.6052, 0.2809, 0.1119, 0.002]),
        'education': rng.choice(['university.degree', 'high.school', 'basic.9y', 'professional.course', 'basic.4y', 'basic.6y', 'unknown', 'illiterate', 'Basic'], n, p=[0.2954, 0.2311, 0.1467, 0.1273, 0.1014, 0.0556, 0.042, 0.0004, 0.0001]),
        'default': rng.choice(['no', 'unknown', 'yes'], n, p=[0.7912, 0.2088, 0.0]),
        'housing': rng.choice(['yes', 'no', 'unknown'], n, p=[0.5238, 0.4522, 0.024]),
        'loan': rng.choice(['no', 'yes', 'unknown', 'n', 'y'], n, p=[0.8241, 0.1517, 0.024, 0.0001, 0.0001]),
    })
    return frame

def rebuild_titanic_train():
    """Rebuild a dataset statistically equivalent to titanic_train.csv (891 rows).

    Same columns, same types, same ranges and category mix, so every cell
    below still runs if the original file is missing."""
    rng = np.random.default_rng(42)
    n = 891
    frame = pd.DataFrame({
        'PassengerId': rng.normal(446, 257.4, n).clip(1, 891).round().astype(int),
        'Survived': rng.choice([0, 1], n, p=[0.6162, 0.3838]),
        'Pclass': rng.choice([1, 2, 3], n, p=[0.2424, 0.2065, 0.5511]),
        'Name': [f"{v}_{i}" for i, v in enumerate(rng.choice(['Braund, Mr. Owen Harris', 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'Heikkinen, Miss. Laina', 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'Allen, Mr. William Henry', 'Moran, Mr. James'], n))],
        'Gender': rng.choice(['male', 'female'], n, p=[0.6476, 0.3524]),
        'Age': rng.normal(29.7, 14.53, n).clip(0.42, 80).round(4),
        'SibSp': rng.choice([0, 1, 2, 3, 4, 5, 8], n, p=[0.6823, 0.2346, 0.0314, 0.018, 0.0202, 0.0056, 0.0079]),
        'Parch': rng.choice([0, 1, 2, 3, 4, 5, 6], n, p=[0.761, 0.1324, 0.0898, 0.0056, 0.0045, 0.0056, 0.0011]),
        'Ticket': [f"{v}_{i}" for i, v in enumerate(rng.choice(['347082', '1601', 'CA. 2343', '3101295', 'CA 2144', '347088'], n))],
        'Fare': rng.normal(32.2, 49.69, n).clip(0, 512.329).round(4),
        'Cabin': [f"{v}_{i}" for i, v in enumerate(rng.choice(['G6', 'C23 C25 C27', 'B96 B98', 'F33', 'E101', 'F2'], n))],
        'Embarked': rng.choice(['S', 'C', 'Q'], n, p=[0.7244, 0.189, 0.0866]),
    })
    frame.loc[rng.choice(n, int(n * 0.1987), replace=False), 'Age'] = np.nan
    frame.loc[rng.choice(n, int(n * 0.7710), replace=False), 'Cabin'] = np.nan
    frame.loc[rng.choice(n, int(n * 0.0022), replace=False), 'Embarked'] = np.nan
    return frame

def rebuild_sales():
    """Exact copy of sales.csv, embedded so this notebook never needs the file."""
    import io
    csv_text = """Invoice ID,Branch,City(nature),Customer type,Gender,Product line,Unit price,Quantity,Profit,Total,Time,Payment,cogs,Total ,gross income,Rating,date_visited
750-67-8428,A,Yangon(Local),Member,Female,Health and beauty,74.69,7.0,26.1415,548.9715,13:08,Ewallet,522.83,4.761904762,26.1415,9.1,02-10-2016
226-31-3081,C,Naypyitaw(Local),Normal,Female,Electronic accessories,15.28,5.0,3.82,80.22,10:29,Cash,76.4,4.761904762,3.82,9.6,26-08-2018
&&&,A,Yangon(Local),Normal,Male,Home and lifestyle,46.33,6.0,16.2155,340.5255,13:23,Credit card,324.31,4.761904762,16.2155,7.4,14-09-2020
123-19-1176,A,Yangon(Local),Member,Male,Health and beauty,58.22,8.0,23.288,489.048,20:33,Ewallet,465.76,4.761904762,23.288,8.4,10-10-2022
373-73-7910,A,Yangon(Local),Normal,,Sports and travel,&&&,7.0,30.2085,634.3785,10:37,Ewallet,604.17,4.761904762,30.2085,5.3,28-09-2019
699-14-3026,C,Naypyitaw(Local),Normal,Male,Electronic accessories,85.39,7.0,29.8865,627.6165,18:30,Ewallet,597.73,4.761904762,29.8865,4.1,03-06-2019
355-53-5943,A,Yangon(Local),Member,Female,Electronic accessories,68.84,,20.652,433.692,14:36,Ewallet,413.04,4.761904762,20.652,5.8,05-06-2015
$$$,C,Naypyitaw(Local),Normal,Female,Home and lifestyle,73.56,10.0,36.78,772.38,11:38,Ewallet,735.6,4.761904762,36.78,8.0,17-03-2016
665-32-9167,A,Yangon(Local),Member,Female,Health and beauty,36.26,2.0,3.626,2356.23,17:15,Credit card,72.52,4.761904762,3.626,7.2,21-02-2018
692-92-5582,B,Mandalay(urban),Member,Female,Food and beverages,54.84,6.0,8.226,172.746,13:27,Credit card,164.52,4.761904762,,5.9,04-10-2012
351-62-0822,B,Mandalay(urban),Member,,Fashion accessories,13.25,4.0,2.896,60.816,18:07,Ewallet,57.92,4.761904762,2.896,4.5,29-01-2020
529-56-3974,B,Mandalay(urban),Member,Male,Electronic accessories,25.51,6.0,5.102,107.142,17:03,Cash,102.04,4.761904762,5.102,6.8,08-06-2019
365-64-0515,A,Naypyitaw(Local),Normal,Female,Electronic accessories,46.95,5.0,11.7375,246.4875,10:25,Ewallet,234.75,4.761904762,11.7375,7.1,21-05-2011
252-56-2699,A,Yangon(Local),Normal,Male,Food and beverages,21.25,10.0,21.595,453.495,16:48,Ewallet,431.9,4.761904762,21.595,8.2,12-10-2014
%%%,A,Yangon(Local),Normal,Female,Health and beauty,71.38,10.0,35.69,8975.36,19:21,Credit card,713.8,4.761904762,35.69,5.7,11-11-2016
299-46-1805,B,Mandalay(urban),Member,Female,Sports and travel,93.72,6.0,28.116,590.436,16:19,Cash,562.32,4.761904762,28.116,4.5,19-03-2014
656-95-9349,A,Yangon(Local),Member,Female,Health and beauty,68.93,,24.1255,506.6355,11:03,Credit card,482.51,4.761904762,24.1255,4.6,11-01-2018
***,A,Yangon(Local),Normal,Male,Sports and travel,72.61,6.0,21.783,457.443,10:39,Credit card,435.66,4.761904762,21.783,6.9,08-07-2015
329-62-1586,A,Yangon(Local),Normal,Male,Food and beverages,54.67,3.0,8.2005,172.2105,18:00,Credit card,164.01,4.761904762,8.2005,8.6,03-06-2014"""
    return pd.read_csv(io.StringIO(csv_text))


def rebuild_customer():
    """Exact copy of customer.csv, embedded so this notebook never needs the file."""
    import io
    csv_text = """Cust_ID,Age,Income,Profession,Marital_Status,Vehicle_Type
C001,24,350000,Student,Single,Bike
C002,28,520000,Software Engineer,Married,Hatchback
C003,35,780000,Doctor,Married,SUV
C004,42,1250000,Business,Married,Sedan
C005,31,610000,Teacher,Single,Scooter
C006,45,980000,Banker,Married,Sedan
C007,29,430000,Sales Executive,Single,Bike
C008,38,890000,Professor,Married,SUV
C009,50,1450000,Business,Married,Luxury Car
C010,27,470000,Accountant,Single,Hatchback
C011,33,720000,Engineer,Married,Sedan
C012,41,860000,Government Employee,Married,SUV
C013,26,390000,Nurse,Single,Scooter
C014,36,810000,Lawyer,Married,Sedan
C015,48,1350000,Entrepreneur,Married,Luxury Car
C016,23,320000,Student,Single,Bike
C017,39,920000,IT Consultant,Married,SUV
C018,44,1120000,Business,Married,Sedan
C019,30,560000,HR Executive,Married,Hatchback
C020,34,690000,Marketing Manager,Single,Sedan
C021,52,1520000,CEO,Married,Luxury Car
C022,37,770000,Teacher,Married,Sedan
C023,25,410000,Graphic Designer,Single,Scooter
C024,40,990000,Data Scientist,Married,SUV
C025,46,1280000,Chartered Accountant,Married,Sedan
C026,32,630000,Pharmacist,Married,Hatchback
C027,29,510000,Civil Engineer,Single,Bike
C028,43,1090000,Professor,Married,SUV
C029,27,450000,Sales Executive,Single,Scooter
C030,35,830000,Software Engineer,Married,Sedan
C031,49,1410000,Business,Married,Luxury Car
C032,38,870000,Lawyer,Married,SUV
C033,24,360000,Student,Single,Bike
C034,31,600000,Nurse,Married,Hatchback
C035,47,1180000,Doctor,Married,SUV
C036,33,680000,Teacher,Single,Scooter
C037,28,540000,Accountant,Married,Hatchback
C038,42,1040000,Engineer,Married,Sedan
C039,36,790000,Data Analyst,Married,SUV
C040,26,430000,Marketing Executive,Single,Bike
C041,53,1650000,Entrepreneur,Married,Luxury Car
C042,39,910000,Government Employee,Married,SUV
C043,34,700000,IT Consultant,Married,Sedan
C044,30,580000,HR Executive,Single,Hatchback
C045,45,1220000,Business,Married,SUV
C046,27,490000,Teacher,Single,Scooter
C047,37,840000,Software Engineer,Married,Sedan
C048,41,970000,Banker,Married,SUV
C049,29,520000,Pharmacist,Single,Hatchback
C050,51,1480000,CEO,Married,Luxury Car"""
    return pd.read_csv(io.StringIO(csv_text))

print("Setup complete. pandas", pd.__version__, "| numpy", np.__version__)

category_encoders not installed - using the built-in equivalents below.
Setup complete. pandas 3.0.2 | numpy 2.4.4


# 03 · Encoding Masterclass
### Every categorical-encoding technique worth knowing, on two real datasets

This notebook consolidates and upgrades **three separate, overlapping
notebooks and their autosave checkpoints** from the original session (all of
which explored the same idea in slightly different, unfinished states) into
one complete, working reference.

**Part A — `data/banking.csv`** (41,199 rows): a quick, focused tour —
the original notebook's exact techniques (`LabelEncoder`, `get_dummies`),
upgraded to run on current library versions, plus the one technique the
original was missing: **ordinal encoding**, for the one column here that
actually has a genuine order.

**Part B — `data/titanic_train.csv`** (891 rows): the full treatment —
feature extraction and splitting from free text, leakage-safe missing-data
handling, a six-method encoding comparison, and a `ColumnTransformer` +
`Pipeline` capstone that trains and evaluates a real baseline model.

**Contents**
- Part A: Bank Marketing — quick encoding tour
  1. Load & audit
  2. Clean
  3. Label encoding, one-hot encoding, ordinal encoding
- Part B: Titanic — full pipeline
  4. Load & audit
  5. Feature extraction & splitting from text
  6. Leakage-safe train/test split
  7. Missing-data handling (naive vs. smart, compared)
  8. Encoding showdown — six methods on the same column
  9. `ColumnTransformer` + `Pipeline` + baseline model
  10. Recap

In [2]:
import re
import numpy as np
import pandas as pd
# (category_encoders is imported in the setup cell, with built-in equivalents if absent)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

RANDOM_STATE = 42

---
# Part A · Bank Marketing dataset — quick encoding tour

`data/banking.csv` is a bank's direct-marketing call log: `age`, `job`,
`marital`, `education`, and three yes/no/unknown flags (`default`, `housing`,
`loan`). Big enough (41k rows) that inefficient row-by-row code would
actually be slow — a good reminder to keep everything vectorised.

In [3]:
bank = load_data('banking.csv', rebuild=rebuild_banking)
print(f"Shape: {bank.shape[0]:,} rows x {bank.shape[1]} columns")
bank.head()

Loaded 'banking.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets
Shape: 41,199 rows x 7 columns


,age,job,marital,education,default,housing,loan
0,44.0,blue-collar,married,basic.4y,unknown,yes,no
1,53.0,technician,married,unknown,no,no,no
2,28.0,management,single,university.degree,no,yes,no
3,39.0,services,married,high.school,no,no,no
4,55.0,retired,married,basic.4y,no,yes,no


## 4A · Audit

Same discipline as Part 1's notebook: look before you touch.

In [4]:
print("Missing values:\n", bank.isnull().sum()[bank.isnull().sum() > 0], "\n")

for col in ["job", "marital", "education", "default", "housing", "loan"]:
    print(f"{col:10s} -> {bank[col].unique().tolist()}")

Missing values:
 age    2
dtype: int64 

job        -> ['blue-collar', 'technician', 'management', 'services', 'retired', 'admin.', 'housemaid', 'unemployed', 'entrepreneur', 'self-employed', 'unknown', 'student']
marital    -> ['married', 'single', 'divorced', 'unknown']
education  -> ['basic.4y', 'unknown', 'university.degree', 'high.school', 'basic.9y', 'professional.course', 'basic.6y', 'illiterate', 'Basic']
default    -> ['unknown', 'no', 'yes']
housing    -> ['yes', 'no', 'unknown']
loan       -> ['no', 'yes', 'unknown', 'n', 'y']


Two concrete issues jump out:

1. **`education`** has both `'basic.6y'` *and* a stray `'Basic'` (capital,
   no duration) — inconsistent capitalisation **and** missing information
   (which of 4y/6y/9y does bare "Basic" mean? We genuinely don't know).
2. **`loan`** mixes `'yes'/'no'` with abbreviated `'y'/'n'` — the same two
   real values, spelled two different ways, which would otherwise be silently
   treated as *four* distinct categories by any encoder.

Both are worth fixing before encoding — an encoder can only be as
trustworthy as the categories you hand it.

In [5]:
print(f"Rows with bare 'Basic' in education : {(bank['education'] == 'Basic').sum()}")
print(f"Rows with 'y'/'n' shorthand in loan  : {(bank['loan'].isin(['y', 'n'])).sum()}")

Rows with bare 'Basic' in education : 2
Rows with 'y'/'n' shorthand in loan  : 9


## 5A · Clean

**On `'Basic'`:** we do *not* guess which duration it means — silently
mapping it to e.g. `basic.6y` would fabricate precision the data doesn't
have. The honest move is to fold it into `'unknown'`, which already means
"education status not reliably known" for this dataset.

**On `'y'`/`'n'`:** unlike `'Basic'`, there's no ambiguity — they
unambiguously mean `'yes'`/`'no'`, just abbreviated. Safe to standardise.

In [6]:
bank_clean = bank.copy()

bank_clean["education"] = bank_clean["education"].replace({"Basic": "unknown"})
bank_clean["loan"] = bank_clean["loan"].replace({"y": "yes", "n": "no"})

# Only 2 of 41,199 rows are missing `age` — small enough to drop outright
# rather than build an imputation strategy for.
bank_clean = bank_clean.dropna(subset=["age"])

print("education uniques now:", bank_clean["education"].unique().tolist())
print("loan uniques now     :", bank_clean["loan"].unique().tolist())
print(f"Rows after dropping the 2 missing-age records: {len(bank_clean):,}")

education uniques now: ['basic.4y', 'unknown', 'university.degree', 'high.school', 'basic.9y', 'professional.course', 'basic.6y', 'illiterate']
loan uniques now     : ['no', 'yes', 'unknown']
Rows after dropping the 2 missing-age records: 41,197


## 6A · Label encoding — the original notebook's technique, upgraded

This reproduces the original notebook's exact approach on `job`. The
mechanics haven't changed; what changed is *how you should read the output*:
`LabelEncoder` assigns codes **alphabetically**, which carries no real order
— fine for a tree-based model, misleading for a linear one that would treat
"11" as literally larger than "2".

In [7]:
label_encoder = LabelEncoder()
bank_clean["job_encoded"] = label_encoder.fit_transform(bank_clean["job"])

dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

{'admin.': 0,
 'blue-collar': 1,
 'entrepreneur': 2,
 'housemaid': 3,
 'management': 4,
 'retired': 5,
 'self-employed': 6,
 'services': 7,
 'student': 8,
 'technician': 9,
 'unemployed': 10,
 'unknown': 11}

In [8]:
# Contrast with frequency encoding: same idea (one numeric column, no
# dimensionality blow-up) but the number now means something — how common
# that job is — rather than an arbitrary alphabetical position.
job_freq = bank_clean["job"].value_counts(normalize=True)
bank_clean["job_freq"] = bank_clean["job"].map(job_freq)
bank_clean[["job", "job_encoded", "job_freq"]].drop_duplicates("job").sort_values("job_freq", ascending=False).head(8)

,job,job_encoded,job_freq
8,admin.,0,0.253052
0,blue-collar,1,0.224628
1,technician,9,0.163726
3,services,7,0.096366
2,management,4,0.070976
4,retired,5,0.041775
25,entrepreneur,2,0.035342
68,self-employed,6,0.034493


## 7A · One-hot encoding — the original notebook's technique, upgraded

The original used `pd.get_dummies(df, columns=['marital'], dtype='int')`.
Still correct in current pandas — the one upgrade worth making is
`drop_first=True`, which avoids the **dummy variable trap**: with all *k*
one-hot columns kept, any one of them is perfectly predictable from the
other *k−1*, which breaks the independence assumption behind linear/logistic
regression coefficients (tree models don't care either way).

In [9]:
bank_clean = pd.get_dummies(bank_clean, columns=["marital"], dtype="int", drop_first=True)
[c for c in bank_clean.columns if c.startswith("marital_")]

['marital_married', 'marital_single', 'marital_unknown']

## 8A · Ordinal encoding — the technique the original notebook never used

`education` is the one column in this dataset with a **genuine order**:
illiteracy < basic schooling (4y < 6y < 9y) < high school < professional
course < university degree. Treating it as nominal (one-hot or arbitrary
label codes) throws that order away for no reason. `OrdinalEncoder` with an
explicit category list preserves it — and `'unknown'` is routed to `NaN`
rather than assigned a false rank, since "not known" isn't a point on the
education scale at all.

In [10]:
education_order = [
    "illiterate", "basic.4y", "basic.6y", "basic.9y",
    "high.school", "professional.course", "university.degree", "unknown",
]

ordinal_encoder = OrdinalEncoder(
    categories=[education_order],
    handle_unknown="use_encoded_value", unknown_value=np.nan,
)

# Encode everything EXCEPT 'unknown' meaningfully; route 'unknown' itself to NaN
# by temporarily hiding it from the encoder's view via a masked copy.
edu_for_encoding = bank_clean[["education"]].where(bank_clean[["education"]] != "unknown")
bank_clean["education_ordinal"] = ordinal_encoder.fit_transform(edu_for_encoding)

bank_clean[["education", "education_ordinal"]].drop_duplicates().sort_values("education_ordinal")

,education,education_ordinal
3059,illiterate,0.0
0,basic.4y,1.0
28,basic.6y,2.0
7,basic.9y,3.0
3,high.school,4.0
23,professional.course,5.0
2,university.degree,6.0
1,unknown,NaN


> A real pipeline would follow this with an imputation step for the `NaN`s
> (e.g. a `SimpleImputer` using the median rank, or leaving them missing for
> a model that natively handles `NaN` such as many gradient-boosting
> libraries) — Part B builds exactly that kind of complete pipeline, end to
> end, on the Titanic data.

### Part A recap

| Column | Method | Why |
|---|---|---|
| `job` | Label encoding *and* frequency encoding | Compared side by side — same shape, different meaning |
| `marital` | One-hot (`drop_first=True`) | Nominal, low cardinality, avoids the dummy-variable trap |
| `education` | Ordinal, explicit order, `unknown → NaN` | The one genuinely ordinal column here |

---
# Part B · Titanic dataset — the full pipeline

`data/titanic_train.csv` (891 passengers) is where the original checkpoint
notebooks were headed but never finished — one dropped all missing rows
outright, another had a `NameError` waiting to happen (`onehot_encoder` used
before it was ever created). We rebuild the whole thing properly, end to
end.

In [11]:
titanic = load_data('titanic_train.csv', rebuild=rebuild_titanic_train)
print(f"Shape: {titanic.shape[0]} rows x {titanic.shape[1]} columns\n")
titanic.head()

Loaded 'titanic_train.csv' from /home/claude/work/build/Python-BDA-Complete-Notes/datasets
Shape: 891 rows x 12 columns



,PassengerId,Survived,Pclass,Name,Gender,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 4B · Audit

In [12]:
missing = titanic.isnull().sum()
missing = missing[missing > 0]
print("Missing values:")
print(missing)
print(f"\n'Cabin' alone is {missing['Cabin'] / len(titanic):.0%} missing — too sparse to impute a")
print("specific value into; better to engineer presence/absence instead (Section 5B).")

Missing values:
Age         177
Cabin       687
Embarked      2
dtype: int64

'Cabin' alone is 77% missing — too sparse to impute a
specific value into; better to engineer presence/absence instead (Section 5B).


## 5B · Feature extraction & splitting from text

None of these steps *learn* anything from the data's statistical
distribution — they're fixed, deterministic rules (a regex, an arithmetic
sum). That means it's safe to run them before the train/test split without
any leakage risk; nothing here is "fit" the way an encoder or an imputer is.

In [13]:
# --- Feature SPLIT: one Name column -> Title + Surname ---------------------
titanic["Title"]   = titanic["Name"].str.extract(r",\s*([^\.]+)\.")
titanic["Surname"] = titanic["Name"].str.split(",").str[0]

titanic[["Name", "Title", "Surname"]].head()

,Name,Title,Surname
0,"Braund, Mr. Owen Harris",Mr,Braund
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,Cumings
2,"Heikkinen, Miss. Laina",Miss,Heikkinen
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,Futrelle
4,"Allen, Mr. William Henry",Mr,Allen


In [14]:
# Raw Title has 17 categories, most with a handful of passengers or fewer —
# consolidate synonyms and rare honorifics, the same "group the long tail"
# strategy from Notebook 2.
print(titanic["Title"].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [15]:
title_synonyms = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
rare_titles = ["Dr", "Rev", "Major", "Col", "Don", "Lady", "Sir", "Capt",
               "the Countess", "Jonkheer"]

titanic["Title"] = titanic["Title"].replace(title_synonyms)
titanic["Title"] = titanic["Title"].where(~titanic["Title"].isin(rare_titles), "Rare")
titanic["Title"].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [16]:
# --- Feature EXTRACTION: arithmetic combinations ---------------------------
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1   # +1 for the passenger
titanic["IsAlone"]    = (titanic["FamilySize"] == 1).astype(int)

titanic[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


In [17]:
# --- Feature EXTRACTION + SPLIT from Cabin: presence flag + deck letter ----
# 77% of Cabin is missing (Section 4B), so rather than impute a specific
# cabin number, extract the two things that ARE recoverable and meaningful:
# (1) whether we know the cabin at all, (2) the deck letter, when we do.
titanic["HasCabin"] = titanic["Cabin"].notna().astype(int)
titanic["Deck"] = titanic["Cabin"].str[0].fillna("Unknown")

titanic["Deck"].value_counts()

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [18]:
# --- Feature SPLIT: Ticket -> prefix + numeric part ------------------------
def split_ticket(ticket: str):
    parts = ticket.rsplit(" ", 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0], parts[1]
    if ticket.isdigit():
        return "NONE", ticket
    return ticket, np.nan

ticket_parts = titanic["Ticket"].apply(split_ticket)
titanic["Ticket_Prefix"] = ticket_parts.apply(lambda t: t[0])
titanic["Ticket_Number"] = ticket_parts.apply(lambda t: t[1])

titanic[["Ticket", "Ticket_Prefix", "Ticket_Number"]].head(6)

,Ticket,Ticket_Prefix,Ticket_Number
0,A/5 21171,A/5,21171
1,PC 17599,PC,17599
2,STON/O2. 3101282,STON/O2.,3101282
3,113803,NONE,113803
4,373450,NONE,373450
5,330877,NONE,330877


## 6B · Leakage-safe train/test split

Everything from here on **estimates something from the data** — a median, a
mode, a category's average survival rate — so from this point forward, train
and test must be kept strictly apart: fit on train, apply to test, never the
other way round. `stratify=y` keeps the ~38% survival rate balanced in both
halves.

In [19]:
feature_cols = ["Pclass", "Gender", "Age", "SibSp", "Parch", "Fare", "Embarked",
                "Title", "FamilySize", "IsAlone", "HasCabin", "Deck"]

X = titanic[feature_cols].copy()
y = titanic["Survived"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Survival rate — train: {y_train.mean():.3f}   test: {y_test.mean():.3f}")

Train: (712, 12)   Test: (179, 12)
Survival rate — train: 0.383   test: 0.385


## 7B · Missing-data handling — naive vs. smart, compared side by side

`Age` is missing for ~20% of passengers. The naive fix is one global number
for everyone. A better one recognises that a "Master" (a young boy, by
Edwardian naming convention) and a 1st-class "Mr" have very different typical
ages — grouping the median by `Title` and `Pclass` uses information already
sitting in the table for free.

In [20]:
naive_median = X_train["Age"].median()

# Fit the LOOKUP TABLE on train only...
age_lookup = X_train.groupby(["Title", "Pclass"])["Age"].median()

def impute_age(frame, lookup, fallback):
    filled = frame["Age"].copy()
    for (title, pclass), median_age in lookup.items():
        mask = filled.isna() & (frame["Title"] == title) & (frame["Pclass"] == pclass)
        filled.loc[mask] = median_age
    return filled.fillna(fallback)   # fallback covers any (Title, Pclass) combo unseen in train

# ...then APPLY it to both train and test, never re-fit on test.
X_train["Age_smart"] = impute_age(X_train, age_lookup, naive_median)
X_test["Age_smart"]  = impute_age(X_test, age_lookup, naive_median)

X_train["Age_naive"] = X_train["Age"].fillna(naive_median)

comparison = X_train.loc[X_train["Age"].isna(), ["Title", "Pclass", "Age_naive", "Age_smart"]].head(8)
comparison

,Title,Pclass,Age_naive,Age_smart
692,Mr,3,28.5,27.0
481,Mr,2,28.5,30.0
527,Mr,1,28.5,40.0
557,Mr,1,28.5,40.0
828,Mr,3,28.5,27.0
846,Mr,3,28.5,27.0
250,Mr,3,28.5,27.0
409,Miss,3,28.5,18.0


In [21]:
print(f"Naive global median used for EVERY missing age : {naive_median:.1f}")
print("Smart, group-aware medians actually used         :")
age_lookup.round(1)

Naive global median used for EVERY missing age : 28.5
Smart, group-aware medians actually used         :


Title   Pclass
Master  1          2.5
        2          1.0
        3          5.0
Miss    1         30.0
        2         24.0
        3         18.0
Mr      1         40.0
        2         30.0
        3         27.0
Mrs     1         39.0
        2         33.5
        3         31.0
Rare    1         49.0
        2         51.0
Name: Age, dtype: float64

In [22]:
# Finish applying the chosen (smart) imputation as the real 'Age' column
X_train["Age"] = X_train["Age_smart"]
X_test["Age"]  = X_test["Age_smart"]
X_train = X_train.drop(columns=["Age_naive", "Age_smart"])
X_test  = X_test.drop(columns=["Age_smart"])

# Embarked: only 2 rows missing in the full dataset — fit the mode on TRAIN only
embarked_mode = X_train["Embarked"].mode().iloc[0]
X_train["Embarked"] = X_train["Embarked"].fillna(embarked_mode)
X_test["Embarked"]  = X_test["Embarked"].fillna(embarked_mode)

print("Remaining missing values in X_train:", X_train.isna().sum().sum())
print("Remaining missing values in X_test :", X_test.isna().sum().sum())

Remaining missing values in X_train: 0
Remaining missing values in X_test : 0


## 8B · Encoding showdown — six methods, one dataset

Six techniques, each fit on **train only**, then applied to test. The
verdict table from Notebook 2 is repeated at the end with Titanic-specific
choices filled in.

In [23]:
# --- (1) Label encoding — binary column, no ordinality risk ----------------
sex_encoder = LabelEncoder()
X_train["Gender_label"] = sex_encoder.fit_transform(X_train["Gender"])
X_test["Gender_label"]  = sex_encoder.transform(X_test["Gender"])
dict(zip(sex_encoder.classes_, range(len(sex_encoder.classes_))))

{'female': 0, 'male': 1}

In [24]:
# --- (2) One-hot via sklearn — nominal, low/medium cardinality -------------
onehot_encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
onehot_encoder.fit(X_train[["Embarked", "Title"]])

onehot_cols = onehot_encoder.get_feature_names_out(["Embarked", "Title"])
onehot_train = pd.DataFrame(onehot_encoder.transform(X_train[["Embarked", "Title"]]),
                             columns=onehot_cols, index=X_train.index)
onehot_test = pd.DataFrame(onehot_encoder.transform(X_test[["Embarked", "Title"]]),
                            columns=onehot_cols, index=X_test.index)
onehot_train.head()

,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
692,0.0,1.0,0.0,1.0,0.0,0.0
481,0.0,1.0,0.0,1.0,0.0,0.0
527,0.0,1.0,0.0,1.0,0.0,0.0
855,0.0,1.0,0.0,0.0,1.0,0.0
801,0.0,1.0,0.0,0.0,1.0,0.0


In [25]:
# --- (3) Frequency encoding — Deck has 9 categories, several tiny ----------
deck_freq = X_train["Deck"].value_counts(normalize=True)
X_train["Deck_freq"] = X_train["Deck"].map(deck_freq)
X_test["Deck_freq"]  = X_test["Deck"].map(deck_freq).fillna(0)   # a deck letter unseen in train -> 0
deck_freq.round(3)

Deck
Unknown    0.775
C          0.058
B          0.048
E          0.041
D          0.037
A          0.020
F          0.015
G          0.006
T          0.001
Name: proportion, dtype: float64

In [26]:
# --- (4) Target (mean) encoding — needs y_train, so it MUST be fit here, --
#     not before the split, or test-set survival outcomes would leak into
#     the encoding of the training features (a classic, very real bug).
#     category_encoders' TargetEncoder smooths small groups toward the
#     overall mean automatically -- watch what happens to Deck 'T' (n=1).
target_encoder = ce.TargetEncoder(cols=["Deck"])
X_train["Deck_target"] = target_encoder.fit_transform(X_train["Deck"], y_train)
X_test["Deck_target"]  = target_encoder.transform(X_test["Deck"])

X_train[["Deck", "Deck_target"]].drop_duplicates().sort_values("Deck_target")

,Deck,Deck_target
692,Unknown,0.293299
339,T,0.348570
647,A,0.409761
251,G,0.416734
715,F,0.563537
527,C,0.624201
763,B,0.655324
835,E,0.662417
218,D,0.689841


In [27]:
# --- (5) Binary encoding — column-count middle ground for higher cardinality
binary_encoder = ce.BinaryEncoder(cols=["Deck"])
deck_binary_train = binary_encoder.fit_transform(X_train["Deck"])
deck_binary_test  = binary_encoder.transform(X_test["Deck"])

print(f"Deck has {X_train['Deck'].nunique()} categories.")
print(f"  One-hot would need : {X_train['Deck'].nunique() - 1} columns (after drop_first)")
print(f"  Binary needs just  : {deck_binary_train.shape[1]} columns")
deck_binary_train.head()

Deck has 9 categories.
  One-hot would need : 8 columns (after drop_first)
  Binary needs just  : 4 columns


,Deck_0,Deck_1,Deck_2,Deck_3
692,0,0,0,1
481,0,0,0,1
527,0,1,0,0
855,0,0,0,1
801,0,0,0,1


In [28]:
# --- (6) Ordinal-by-construction — Pclass is ALREADY a clean 1/2/3 ranking -
# The lesson here is different: sometimes the best "encoding" is recognising
# a column is already numeric and ordinal, and leaving it alone rather than
# one-hot-exploding it into three columns for no benefit.
X_train["Pclass"].unique()

array([3, 2, 1])

| # | Method | Fit on | Titanic column used | Verdict |
|---|--------|--------|---------------------|---------|
| 1 | Label encoding | train | `Gender` (binary) | Safe — only 2 classes, no false order implied |
| 2 | One-hot (sklearn) | train | `Embarked`, `Title` | Nominal, low cardinality — the default choice |
| 3 | Frequency encoding | train | `Deck` | One numeric column instead of 8 sparse ones |
| 4 | Target encoding (smoothed) | train **+ y_train** | `Deck` | Powerful, but the #1 leakage risk if fit before the split |
| 5 | Binary encoding | train | `Deck` | 4 columns instead of 8 — a middle ground |
| 6 | "Leave it numeric" | — | `Pclass` | Already ordinal — encoding it further would only hurt |

## 9B · `ColumnTransformer` + `Pipeline` — the professional pattern, end to end

Manually fitting six encoders and remembering to apply each one to both train
and test is exactly the kind of bookkeeping that's easy to get wrong under
deadline pressure. `ColumnTransformer` bundles "which columns get which
transformation" into a single object; `Pipeline` chains that with a model.
Calling `.fit()` once on train and `.predict()` on test makes the leakage-safe
discipline from Sections 6B–8B **automatic** instead of manual.

In [29]:
numeric_features     = ["Age", "Fare", "FamilySize", "IsAlone", "HasCabin"]
categorical_features  = ["Pclass", "Gender", "Embarked", "Title"]

preprocessor = ColumnTransformer(transformers=[
    ("numeric",     StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
])

titanic_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

titanic_pipeline.fit(X_train[numeric_features + categorical_features], y_train)
predictions = titanic_pipeline.predict(X_test[numeric_features + categorical_features])

print(f"Held-out test accuracy: {accuracy_score(y_test, predictions):.3f}\n")
print(classification_report(y_test, predictions, target_names=["Did not survive", "Survived"]))

Held-out test accuracy: 0.827

                 precision    recall  f1-score   support

Did not survive       0.84      0.88      0.86       110
       Survived       0.80      0.74      0.77        69

       accuracy                           0.83       179
      macro avg       0.82      0.81      0.81       179
   weighted avg       0.83      0.83      0.83       179



That accuracy is the payoff for everything above it: it's only reachable
because `Age` was sensibly imputed, `Title`/`Deck`/`FamilySize` were
engineered out of raw text, and every encoder was fit on train and merely
*applied* to test — change any one of those and the number moves.

## 10 · Recap

**Part A (Bank Marketing):**
- Reproduced the original notebook's `LabelEncoder` + `get_dummies` pattern
  on current library versions, `drop_first=True` added to avoid the dummy
  variable trap.
- Added the one thing the original never tried: `OrdinalEncoder` with an
  explicit, meaningful order for `education` — with `'unknown'` routed to
  `NaN` rather than assigned a fabricated rank.

**Part B (Titanic):**
- Rebuilt what the abandoned checkpoint notebooks were reaching for, and
  finished it: text-based feature extraction and splitting (`Title`,
  `Surname`, `Deck`, `Ticket_Prefix`), leakage-safe train/test discipline,
  a six-method encoding comparison, and a full `ColumnTransformer` +
  `Pipeline` + baseline model that actually runs end to end.

**The throughline across all three notebooks in this session:** look at the
data before you touch it, extract everything a single column is quietly
hiding, match the encoding to whether the category is ordinal or nominal —
and prove every claim by running the code, not just asserting it.

---

## Exam quick-reference

| To do this | Write this |
|---|---|
| Label | `s.astype('category').cat.codes` |
| Ordinal with a real order | `s.map({'Low':1,'High':3})` |
| One-hot | `pd.get_dummies(s, dtype=int)` |
| Dummy | `pd.get_dummies(s, drop_first=True)` |
| Frequency | `s.map(s.value_counts(normalize=True))` |
| Target (train only!) | `s.map(train.groupby(s)['y'].mean())` |
| Smoothed target | `(n*mean + k*prior) / (n + k)` |
| Column count check | `encoded.shape[1]` |

### Adapting this in the exam

- High-cardinality column in an exam? Say why one-hot is unsuitable, then use frequency or smoothed target encoding.
- Always state the nominal/ordinal judgement before showing any code — that's where the mark is.

### Traps that cost marks

- Target encoding fitted on the whole dataset leaks the answer and inflates your score. Fit on train only.
- Smoothing matters: a category with two rows shouldn't get its raw mean. Blend it toward the overall mean.
- Binary and hashing encoding aren't reversible — you can't recover the original label.
- Whatever you choose, the encoded columns must be produced identically for train and test.